This is the Kaggle dataset

Imports & Relative Path Setup

In [ ]:
import os
import random
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import Dataset
from PIL import Image
import pandas as pd

# --- CONFIGURATION ---
# CVC-ClinicDB uses PNG/Original for images and PNG/Ground Truth for masks
SOURCE_ROOT = Path("PNG")
IMG_DIR     = SOURCE_ROOT / "Original"
MASK_DIR    = SOURCE_ROOT / "Ground Truth"
VALID_EXTS  = {'.jpg', '.jpeg', '.png'}

# Split CSV output
SPLIT_CSV = SOURCE_ROOT / "splits.csv"

# Split Ratios
TRAIN_RATIO = 0.8
VAL_RATIO   = 0.1
TEST_RATIO  = 0.1
SEED        = 42

print(f"🔧 Project Root : {Path.cwd()}")
print(f"🖼️  Images       : {IMG_DIR}")
print(f"🎭 Masks         : {MASK_DIR}")
print(f"📄 Split file   : {SPLIT_CSV}")

Verify Data

In [ ]:
def verify_raw_data():
    """Checks if raw CVC-ClinicDB data is valid before we split it."""
    if not IMG_DIR.exists() or not MASK_DIR.exists():
        print(f"⚠️ Raw data not found.")
        print(f"   Expected images at : {IMG_DIR}")
        print(f"   Expected masks at  : {MASK_DIR}")
        print("   Download from: https://www.kaggle.com/datasets/balraj98/cvcclinicdb")
        return False

    images = sorted([f.name for f in IMG_DIR.iterdir()
                     if f.suffix.lower() in VALID_EXTS and f.name != '.DS_Store'])
    masks  = sorted([f.name for f in MASK_DIR.iterdir()
                     if f.suffix.lower() in VALID_EXTS and f.name != '.DS_Store'])

    print(f"📊 Found {len(images)} images and {len(masks)} masks")

    # 1. Count Check
    if len(images) != len(masks):
        print(f"❌ Mismatch: {len(images)} images vs {len(masks)} masks")
        return False

    # 2. Spot Check (First 5 pairs)
    print("🔍 Verifying alignment on first 5 samples...")
    for img_name in images[:5]:
        img_p  = str(IMG_DIR  / img_name)
        mask_p = str(MASK_DIR / img_name)

        if not (MASK_DIR / img_name).exists():
            print(f"❌ No matching mask for: {img_name}")
            return False

        img  = cv2.imread(img_p)
        mask = cv2.imread(mask_p)
        if img is None or mask is None:
            print(f"❌ Could not read: {img_name}")
            return False

        if img.shape[:2] != mask.shape[:2]:
            print(f"❌ Dimension mismatch: {img_name} — image {img.shape[:2]} vs mask {mask.shape[:2]}")
            return False

    print("✅ Raw data looks good!")
    return True

verify_raw_data()

Rotate, Blur, Brighten, and Split Data

In [ ]:
def augment_raw_dataset(force=False):
    """Create rotated/brightened/blurred augmentations before splitting."""
    augmented_root = SOURCE_ROOT / "augmented"
    aug_img_dir    = augmented_root / "images"
    aug_mask_dir   = augmented_root / "masks"

    if augmented_root.exists() and not force:
        print(f"ℹ️ Augmented dataset already exists at {augmented_root}")
        return augmented_root

    if not IMG_DIR.exists() or not MASK_DIR.exists():
        raise FileNotFoundError(f"Source data not found. Run verify_raw_data() first.")

    aug_img_dir.mkdir(parents=True, exist_ok=True)
    aug_mask_dir.mkdir(parents=True, exist_ok=True)

    print(f"🔧 Generating augmented data under {augmented_root}...")
    rng = random.Random(SEED)

    for img_path in sorted(IMG_DIR.iterdir()):
        if img_path.suffix.lower() not in VALID_EXTS:
            continue

        stem = img_path.stem
        mask_candidates = list(MASK_DIR.glob(f"{stem}.*"))
        if not mask_candidates:
            print(f"⚠️ Skipping {stem}: no matching mask found")
            continue
        mask_path = mask_candidates[0]

        img  = cv2.imread(str(img_path))
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if img is None or mask is None:
            print(f"⚠️ Failed to load {stem}")
            continue

        # Save original pair
        cv2.imwrite(str(aug_img_dir  / f"{stem}{img_path.suffix}"), img)
        cv2.imwrite(str(aug_mask_dir / f"{stem}{mask_path.suffix}"), mask)

        h, w   = img.shape[:2]
        center = (w / 2.0, h / 2.0)

        # 3 augmented variants per image
        for i in range(1, 4):
            angle             = rng.uniform(0, 360)
            brightness_factor = rng.uniform(0.7, 1.3)
            blur_strength     = rng.choice([3, 5, 7, 9])

            blurred   = cv2.GaussianBlur(img, (blur_strength, blur_strength), 0)
            brightened = np.clip(blurred.astype(np.float32) * brightness_factor, 0, 255).astype(np.uint8)

            rot_mat  = cv2.getRotationMatrix2D(center, angle, 1.0)
            aug_img  = cv2.warpAffine(brightened, rot_mat, (w, h),
                                      flags=cv2.INTER_LINEAR,  borderMode=cv2.BORDER_REFLECT)
            aug_mask = cv2.warpAffine(mask,       rot_mat, (w, h),
                                      flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_REFLECT)

            suffix = f"_aug{i}_r{int(angle)}_b{int(brightness_factor*100)}_blur{blur_strength}"
            cv2.imwrite(str(aug_img_dir  / f"{stem}{suffix}{img_path.suffix}"),  aug_img)
            cv2.imwrite(str(aug_mask_dir / f"{stem}{suffix}{mask_path.suffix}"), aug_mask)

    print("✅ Augmentation complete")
    return augmented_root


def create_split(source_root):
    """Generate splits.csv once (frozen). Subsequent calls load the existing file."""
    split_csv = source_root / "splits.csv"

    if split_csv.exists():
        print(f"✅ Split already frozen at '{split_csv}'. Loading...")
        df = pd.read_csv(split_csv)
        print(df['split'].value_counts().to_string())
        return df, split_csv

    img_dir  = source_root / "images"
    mask_dir = source_root / "masks"

    img_stems  = [p.stem for p in img_dir.iterdir()  if p.suffix.lower() in VALID_EXTS]
    mask_stems = {p.stem for p in mask_dir.iterdir() if p.suffix.lower() in VALID_EXTS}
    paired     = [stem for stem in img_stems if stem in mask_stems]

    rng = random.Random(SEED)
    rng.shuffle(paired)

    n       = len(paired)
    n_train = int(n * TRAIN_RATIO)
    n_val   = int(n * VAL_RATIO)
    tags    = ['train'] * n_train + ['val'] * n_val + ['test'] * (n - n_train - n_val)

    rows = []
    for stem, split in zip(paired, tags):
        img_file  = next(img_dir.glob(f"{stem}.*"))
        mask_file = next(mask_dir.glob(f"{stem}.*"))
        rows.append({
            'stem':       stem,
            'image_path': str(img_file.relative_to(source_root)),
            'mask_path':  str(mask_file.relative_to(source_root)),
            'split':      split,
        })

    df = pd.DataFrame(rows)
    df.to_csv(split_csv, index=False)
    print(f"💾 Split frozen → {split_csv}")
    print(df['split'].value_counts().to_string())
    return df, split_csv


# Run augmentation then freeze the split
augmented_root = augment_raw_dataset()
splits_df, SPLIT_CSV = create_split(augmented_root)

Dataset Class

In [ ]:
class CVCClinicDataset(Dataset):
    def __init__(self, split="train", transform=None):
        self.root      = augmented_root
        self.split     = split
        self.transform = transform

        if not SPLIT_CSV.exists():
            raise FileNotFoundError("splits.csv not found — run create_split() first.")

        df  = pd.read_csv(SPLIT_CSV)
        sub = df[df['split'] == split].reset_index(drop=True)

        self.image_paths = [self.root / p for p in sub['image_path']]
        self.mask_paths  = [self.root / p for p in sub['mask_path']]
        self.stems       = sub['stem'].tolist()

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = cv2.imread(str(self.image_paths[idx]))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask  = cv2.imread(str(self.mask_paths[idx]), cv2.IMREAD_GRAYSCALE)

        _, mask = cv2.threshold(mask, 127, 1.0, cv2.THRESH_BINARY)
        mask = mask.astype(np.float32)

        return image, mask

    def raw(self, idx):
        image = cv2.cvtColor(cv2.imread(str(self.image_paths[idx])), cv2.COLOR_BGR2RGB)
        mask  = cv2.imread(str(self.mask_paths[idx]), cv2.IMREAD_GRAYSCALE)
        return image, mask


train_ds = CVCClinicDataset(split="train")
val_ds   = CVCClinicDataset(split="val")
test_ds  = CVCClinicDataset(split="test")

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

Confirm Split is Frozen (no leakage)

In [ ]:
def check_split_frozen():
    df    = pd.read_csv(SPLIT_CSV)
    train = set(df[df.split == 'train'].stem)
    val   = set(df[df.split == 'val'].stem)
    test  = set(df[df.split == 'test'].stem)

    checks = {'train∩val': train & val, 'train∩test': train & test, 'val∩test': val & test}
    all_ok = True
    for name, overlap in checks.items():
        if overlap:
            print(f"❌ Overlap in {name}: {len(overlap)} items → {list(overlap)[:3]}")
            all_ok = False
        else:
            print(f"✅ No overlap: {name}")

    total = len(df)
    print(f"\nCounts:    {df.split.value_counts().to_dict()}")
    print(f"Fractions: train={len(train)/total:.1%}  val={len(val)/total:.1%}  test={len(test)/total:.1%}")
    print(f"\n{'✅ Split is clean and frozen.' if all_ok else '❌ Fix overlaps before training!'}")

check_split_frozen()

Mask alignment and Resolution Consistency

In [ ]:
def check_resolution_consistency(sample_n=0):
    """
    Checks every (or sample_n random) pairs for:
      - image and mask having the same H×W
      - mask pixels being only {0, 255}
    """
    df = pd.read_csv(SPLIT_CSV)
    if sample_n:
        df = df.sample(min(sample_n, len(df)), random_state=SEED)

    mismatched, bad_vals, res_counts = [], [], {}

    for _, row in df.iterrows():
        img  = cv2.imread(str(augmented_root / row.image_path))
        mask = cv2.imread(str(augmented_root / row.mask_path), cv2.IMREAD_GRAYSCALE)

        ih, iw = img.shape[:2]
        mh, mw = mask.shape[:2]
        res_counts[(iw, ih)] = res_counts.get((iw, ih), 0) + 1

        if (ih, iw) != (mh, mw):
            mismatched.append((row.stem, (iw, ih), (mw, mh)))

        unique = set(mask.flatten().tolist())
        if not unique.issubset({0, 255}):
            bad_vals.append((row.stem, unique))

    print(f"Checked {len(df)} pairs\n")

    if mismatched:
        print(f"❌ {len(mismatched)} resolution mismatches:")
        for s, i, m in mismatched[:5]:
            print(f"   {s}: image={i}  mask={m}")
    else:
        print("✅ All image/mask pairs share the same resolution.")

    if bad_vals:
        print(f"⚠️  {len(bad_vals)} masks with unexpected pixel values (not {{0,255}}):")
        for s, v in bad_vals[:5]:
            print(f"   {s}: {v}")
    else:
        print("✅ All mask pixels are {0, 255}.")

    print(f"\nTop-5 resolutions (W×H):")
    for (w, h), cnt in sorted(res_counts.items(), key=lambda x: -x[1])[:5]:
        print(f"  {w}×{h} → {cnt} images")

check_resolution_consistency(sample_n=0)  # 0 = check all pairs

Overlay Sanity Check

In [ ]:
def overlay_sanity_check(dataset, n_samples=6, seed=42):
    """
    Plots image | mask | green overlay for random samples.
    If green doesn't sit on the polyp → alignment is broken.
    """
    rng  = random.Random(seed)
    idxs = rng.sample(range(len(dataset)), min(n_samples, len(dataset)))

    fig, axes = plt.subplots(n_samples, 3, figsize=(13, 4.2 * n_samples))
    if n_samples == 1:
        axes = [axes]

    for row, idx in enumerate(idxs):
        image, mask = dataset.raw(idx)
        stem        = dataset.stems[idx]

        if image.shape[:2] != mask.shape[:2]:
            mask = cv2.resize(mask, (image.shape[1], image.shape[0]),
                              interpolation=cv2.INTER_NEAREST)

        # Build green overlay
        overlay           = image.copy().astype(float)
        poly              = mask == 255
        overlay[poly, 0]  = overlay[poly, 0] * 0.4
        overlay[poly, 1]  = np.clip(overlay[poly, 1] * 0.4 + 180, 0, 255)
        overlay[poly, 2]  = overlay[poly, 2] * 0.4
        overlay           = overlay.astype(np.uint8)

        h, w = image.shape[:2]
        axes[row][0].imshow(image);          axes[row][0].set_title(f"{stem}\n{w}×{h}", fontsize=8); axes[row][0].axis('off')
        axes[row][1].imshow(mask, cmap='gray'); axes[row][1].set_title("Mask | polyp");              axes[row][1].axis('off')
        axes[row][2].imshow(overlay);        axes[row][2].set_title("Overlay (green = polyp)", fontsize=8); axes[row][2].axis('off')

    plt.suptitle(f"Mask Alignment — {dataset.split} split", fontsize=13, fontweight='bold', y=1.005)
    plt.tight_layout()
    out = SOURCE_ROOT / f"overlay_check_{dataset.split}.png"
    plt.savefig(out, dpi=120, bbox_inches='tight')
    plt.show()
    print(f"✅ Saved → {out}")

overlay_sanity_check(train_ds, n_samples=6, seed=SEED)
overlay_sanity_check(val_ds,   n_samples=4, seed=SEED)
overlay_sanity_check(test_ds,  n_samples=4, seed=SEED)